# Detection: the no-XBRL ablation, and all three Channel A variants evaluated

RQ1's main ablation: Channel A trained without the XBRL-mined negatives. It reuses the full and no-slot models from the detector_train notebook (attach that notebook's output as an input), trains the no-XBRL model, and evaluates all three against Channel B on the same 2,000 development items. Expect about 4 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Models from the detector_train notebook, and the training data

In [ ]:
import glob, os
found = {name: sorted(glob.glob(f'/kaggle/input/**/{name}/model', recursive=True)) for name in ('channel-a', 'channel-a-noslot')}
if not all(found.values()):
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.count(os.sep) <= 5:
            print(root, sorted(dirs)[:8], sorted(files)[:8])
    raise RuntimeError('Attach the detector_train notebook\'s output as an input; its models were not found under /kaggle/input')
A, NOSLOT = found['channel-a'][0], found['channel-a-noslot'][0]
print(A, NOSLOT)
CHECKPOINT = 'KRLabsOrg/lettucedect-v2-mmbert-base'
!faithguard detector-data

## Ablation: without XBRL negatives

In [ ]:
!python -m faithguard.train.detector --data runs/detector/train.jsonl --checkpoint $CHECKPOINT --exclude xbrl --out {OUT}/channel-a-noxbrl

## Evaluate all three (also writes per-claim rows for the A+B fusion)

In [ ]:
!python -m faithguard.train.detector_eval --model-dir $A --out {OUT}/eval-channel-a
!python -m faithguard.train.detector_eval --model-dir $NOSLOT --out {OUT}/eval-channel-a-noslot
!python -m faithguard.train.detector_eval --model-dir {OUT}/channel-a-noxbrl/model --out {OUT}/eval-channel-a-noxbrl

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])